# Checking BDT Input Variables for Redundancy

Builds the list of BDT input variables that add no information and should be excluded from training
(paste the printed list into `redundant_bdt_vars` in `src/ntuple_variables/variables.py`).

All checks run on the full `presel_df_train_vars.parquet`, on the values **as the BDT sees them**: cast to float32, with
non-finite values and |x| > `FEATURE_MISSING_THRESHOLD` turned into missing (`bdt_feature_matrix` in `src/bdt_training_setup.py`).

A variable B is dropped when one of these holds:
1. **Constant**: B takes a single value (or is always missing) on every event.
2. **Order-equivalent**: B is a strictly monotonic (increasing or decreasing) function of a kept variable A, with missing values in the same events.
   Exact duplicates are the special case B = A. Tree splits depend only on the ordering of a variable, and XGBoost's `hist` binning
   uses quantiles, so B and A give identical splits.
3. **Conditional copy**: on every event B either equals a kept variable A or takes one fixed default value, and the events with the
   default are exactly picked out by the value of a kept discrete variable F (e.g. a `_filled` flag). Then B = (F ? A : default),
   which the BDT can rebuild with a split on F followed by splits on A. B must copy at least two distinct values of A, so
   coincidences like a 0/1 flag agreeing with an energy that is 0 on the same events don't count.

Variables used to justify a drop (A and F) are protected from being dropped themselves, so every drop stays justified by kept variables.
Variables in `not_using_for_training_vars` are excluded for other reasons and are never used as A or F.

The informational tables at the end list variables that are *mostly* redundant but carry some unique information, so they are kept.

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from tqdm.notebook import tqdm
import hashlib
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))  # bdt_training_setup uses src-relative imports
from src.file_locations import intermediate_files_location
from src.bdt_training_setup import FEATURE_MISSING_THRESHOLD

from src.ntuple_variables.variables import (combined_training_vars, combined_training_vars_with_duplicates,
                                            redundant_bdt_vars, not_using_for_training_vars)

pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)

In [ ]:
path = f"{intermediate_files_location}/presel_df_train_vars.parquet"
print(path, "modified", pd.Timestamp(os.path.getmtime(path), unit="s"))
available = set(pl.scan_parquet(path).collect_schema().names())
print("training vars missing from file:", [v for v in combined_training_vars_with_duplicates if v not in available])
print("not_using_for_training_vars entries that aren't training variables (typos?):",
      [v for v in not_using_for_training_vars if v not in combined_training_vars_with_duplicates])

# candidate variables, in priority order for which one to keep: currently-used variables first, then the rest, each in list order
universe = [v for v in combined_training_vars_with_duplicates if v in available and v not in not_using_for_training_vars]
currently_used = set(combined_training_vars)
universe = [v for v in universe if v in currently_used] + [v for v in universe if v not in currently_used]
priority = {v: i for i, v in enumerate(universe)}
print(len(universe), "candidate variables")

raw = pl.read_parquet(path, columns=universe)
def bdt_view(c):
    x = pl.col(c).cast(pl.Float64)
    return pl.when(x.abs() <= FEATURE_MISSING_THRESHOLD).then(x).otherwise(None).cast(pl.Float32).alias(c)
df = raw.select([bdt_view(c) for c in universe])  # missing = null
del raw
n_events = df.height
print(df.shape)

In [ ]:
MISSING = np.float32(-3.0e38)  # stand-in for missing in equality tests (no real value is below -1e10 after preprocessing)

def col(v):
    return df.get_column(v).fill_null(MISSING).to_numpy()

stats = df.select(
    [pl.col(c).n_unique().alias(c) for c in universe]
).row(0, named=True)
n_unique = {c: stats[c] for c in universe}  # null counts as one value
discrete_vars = [c for c in universe if n_unique[c] <= 32]
print(len(discrete_vars), "discrete variables (<= 32 distinct values) usable as flags F")

## 1. Constant variables

In [ ]:
dropped = {}      # var -> dict(reason, kept_equivalent, flag)
protected = set()

for v in universe:
    if n_unique[v] <= 1:
        dropped[v] = dict(reason="constant", equivalent_to="", flag="",
                          detail=f"always {df.get_column(v).drop_nulls().unique().to_list() or 'missing'}")
print(f"{len(dropped)} constant:", list(dropped))

## 2. Order-equivalent variables (including exact duplicates)

Each variable is replaced by its dense rank (missing stays missing) and hashed; reversing the ranks catches decreasing transforms.

In [ ]:
def rank_hashes(batch):
    ranks = df.select([pl.col(c).rank("dense").alias(c) for c in batch])
    out = {}
    for c in batch:
        r = ranks.get_column(c)
        mx = r.max() or 0
        inc = r.fill_null(0).to_numpy().astype(np.int64)
        dec = np.where(inc > 0, mx + 1 - inc, 0)
        out[c] = min(hashlib.sha1(np.ascontiguousarray(inc)).hexdigest(), hashlib.sha1(np.ascontiguousarray(dec)).hexdigest())
    return out

remaining = [v for v in universe if v not in dropped]
rank_key = {}
for i in tqdm(range(0, len(remaining), 64)):
    rank_key.update(rank_hashes(remaining[i:i+64]))

order_groups = {}
for v in remaining:
    order_groups.setdefault(rank_key[v], []).append(v)
order_groups = [sorted(g, key=priority.get) for g in order_groups.values() if len(g) > 1]

for g in order_groups:
    keep = g[0]
    protected.add(keep)
    a = col(keep)
    for v in g[1:]:
        identical = np.array_equal(a, col(v))
        dropped[v] = dict(reason="identical" if identical else "monotonic transform", equivalent_to=keep, flag="", detail="")
print(f"{len(order_groups)} order-equivalent groups, {sum(len(g) - 1 for g in order_groups)} variables dropped")
for g in order_groups:
    print(f"  keep {g[0]:<45} drop {g[1:]}")

## 3. Conditional copies

Candidates come from a 100k-event subsample: ordered pairs (A, B) where B takes a single value on all events where B != A.
Each candidate is then verified on the full dataset, and the set of events where B != A must be exactly determined by the value
of a kept discrete variable F (every F value has either all or none of its events in that set).

In [ ]:
remaining = [v for v in universe if v not in dropped]
rng = np.random.default_rng(0)
sample_idx = np.sort(rng.choice(n_events, size=min(100_000, n_events), replace=False))
Z = np.column_stack([col(v)[sample_idx] for v in tqdm(remaining)])

candidates = []
for i in tqdm(range(len(remaining))):
    neq = Z[:, [i]] != Z                       # (events, vars): B != A, for A = remaining[i]
    n_neq = neq.sum(axis=0)
    hi = np.where(neq, Z, -np.inf).max(axis=0)
    lo = np.where(neq, Z, np.inf).min(axis=0)
    hi_eq = np.where(neq, -np.inf, Z).max(axis=0)
    lo_eq = np.where(neq, np.inf, Z).min(axis=0)
    # B differs from A somewhere and is constant there, and where B == A it takes at least two distinct values
    ok = (n_neq > 0) & (n_neq < len(Z)) & (hi == lo) & (hi_eq > lo_eq)
    ok[i] = False
    for j in np.nonzero(ok)[0]:
        candidates.append((remaining[i], remaining[j]))
del Z
print(len(candidates), "candidate (A, B) pairs from the subsample")

In [ ]:
codes = {}
def flag_codes(f):
    if f not in codes:
        codes[f] = df.get_column(f).rank("dense").fill_null(0).to_numpy().astype(np.int64)
    return codes[f]

def determines(c, mask):
    tot = np.bincount(c)
    in_d = np.bincount(c, weights=mask)
    return np.all((in_d == 0) | (in_d == tot))

flag_cache = {}
def find_flag(d_mask, exclude):
    # a kept discrete variable whose value determines d_mask exactly; prefer the fewest distinct values.
    # flags are first screened on the subsample, then confirmed on all events
    key = hashlib.sha1(np.packbits(d_mask)).hexdigest() + "|" + ",".join(sorted(exclude))
    if key in flag_cache and (flag_cache[key] is None or flag_cache[key] not in dropped):
        return flag_cache[key]
    d_sample = d_mask[sample_idx]
    result = None
    for f in sorted(discrete_vars, key=lambda f: (n_unique[f], priority[f])):
        if f in exclude or f in dropped:
            continue
        c = flag_codes(f)
        if not determines(c[sample_idx], d_sample):
            continue
        if determines(c, d_mask):
            result = f
            break
    flag_cache[key] = result
    return result

# drop the least-informative B first (smallest number of events where it equals A and is not its default)
verified = []
for a, b in tqdm(candidates):
    av, bv = col(a), col(b)
    d_mask = av != bv
    if not d_mask.any() or np.unique(bv[d_mask]).size != 1 or np.unique(bv[~d_mask]).size < 2:
        continue
    verified.append((a, b, d_mask, bv[d_mask][0]))
print(len(verified), "candidates hold on the full dataset")

cond_rows = []
for a, b, d_mask, default in sorted(verified, key=lambda t: (-t[2].sum(), priority[t[1]])):
    if b in dropped or a in dropped or b in protected:
        continue
    f = find_flag(d_mask, exclude={a, b})
    if f is None:
        cond_rows.append(dict(var=b, copies=a, frac_differ=d_mask.mean(), result="kept: no flag determines where it differs"))
        continue
    protected.update([a, f])
    default_str = "missing" if default == MISSING else f"{default:g}"
    dropped[b] = dict(reason="conditional copy", equivalent_to=a, flag=f,
                      detail=f"= {a} except = {default_str} on {d_mask.mean():.1%} of events, set fixed by {f}")
    cond_rows.append(dict(var=b, copies=a, frac_differ=d_mask.mean(), result=f"dropped (flag {f}, default {default_str})"))
print(pd.DataFrame(cond_rows).to_string())

## Report: variables to exclude from BDT training

In [ ]:
report = pd.DataFrame([dict(var=v, **info) for v, info in dropped.items()])
report["currently_used"] = report["var"].isin(currently_used)
report["_p"] = report["var"].map(priority)
report = report.sort_values(["reason", "_p"]).drop(columns="_p").reset_index(drop=True)
print(report.to_string())
report.to_csv("../plots/bdt_var_redundancy_report.csv", index=False)

kept = [v for v in universe if v not in dropped]
print()
print(f"{len(universe)} candidates -> {len(kept)} kept, {len(dropped)} excluded")
print(f"  {report.currently_used.sum()} of the excluded are currently used for training")
lost = [v for v in combined_training_vars if v in dropped]
newly_used = sorted([v for v in kept if v not in currently_used], key=priority.get)
print("currently used, would be excluded:", lost)
print("currently excluded, would be used:", newly_used)

In [ ]:
# paste into src/ntuple_variables/variables.py, replacing redundant_bdt_vars
print("# output from checking_bdt_variables.ipynb (see plots/bdt_var_redundancy_report.csv)")
print("redundant_bdt_vars = [")
for v in sorted(dropped, key=lambda v: combined_training_vars_with_duplicates.index(v)):
    info = dropped[v]
    why = info["reason"] + (f" of {info['equivalent_to']}" if info["equivalent_to"] else "") + (f" given {info['flag']}" if info["flag"] else "")
    print(f"    '{v}',  # {why}")
print("]")

## Informational: mostly-redundant variables that are kept

**Partial copies.** Pairs where B equals A on at least 80% of the events where B is not at its most common value
(`frac_B_info_shared`), but B can't be rebuilt from A plus a flag, so B is kept. `frac_events_B_differs` is the fraction of
all events where B != A; these events are where B carries information A doesn't (e.g. B at its default while A is filled,
or B filled while A is at its default).

In [ ]:
remaining = kept
sample = np.column_stack([col(v)[sample_idx] for v in remaining])
mode_val = np.array([np.unique(sample[:, j], return_counts=True)[0][np.argmax(np.unique(sample[:, j], return_counts=True)[1])]
                     for j in range(sample.shape[1])])
informative = sample != mode_val
n_inf = informative.sum(axis=0)
rows = []
for i in tqdm(range(len(remaining))):
    eq_inf = (sample[:, [i]] == sample) & informative
    frac = np.divide(eq_inf.sum(axis=0), n_inf, out=np.zeros(len(remaining)), where=n_inf > 0)
    for j in np.nonzero((frac >= 0.8) & (n_inf >= 50))[0]:
        if j == i or np.unique(sample[eq_inf[:, j], j]).size < 2:  # skip 0/1 flags agreeing by coincidence
            continue
        rows.append(dict(B=remaining[j], A=remaining[i], frac_B_info_shared=frac[j],
                         frac_events_B_differs=np.mean(sample[:, i] != sample[:, j])))
partial = pd.DataFrame(rows).sort_values("frac_B_info_shared", ascending=False).reset_index(drop=True)
print(partial.to_string())

**Near order-equivalence.** Kept pairs with |Spearman rank correlation| ≥ 0.9999 on a 200k-event subsample: nearly the same
ordering, but not exactly, so neither is dropped. Missing values rank below all real values.

In [ ]:
idx = np.sort(rng.choice(n_events, size=min(200_000, n_events), replace=False))
X = np.column_stack([col(v)[idx] for v in remaining]).astype(np.float64)
R = rankdata(X, axis=0)
del X
R -= R.mean(axis=0)
std = R.std(axis=0)
std[std == 0] = 1
R /= std
rho = (R.T @ R) / R.shape[0]
del R
iu = np.triu_indices(len(remaining), k=1)
m = np.abs(rho[iu]) >= 0.9999
near = pd.DataFrame({"var_1": np.array(remaining)[iu[0][m]], "var_2": np.array(remaining)[iu[1][m]], "spearman": rho[iu][m]})
print(near.sort_values("spearman", key=np.abs, ascending=False).to_string())